In [18]:
import os
import json
import numpy as np

# ============================================================
# Paths
# ============================================================
STD_NPZ = "Prepared/Kia_human_train_test_chunks.npz"       # for GRU / L-CNN
LIPAR_NPZ = "Prepared/Kia_LiPar_train_test_chunks.npz"     # for LiPar
OUT_DIR = "Prepared/matched_pairs"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# Loaders
# ============================================================
def load_standard_npz(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    required = ["X_test", "y_test"]
    for k in required:
        if k not in data.files:
            raise KeyError(f"{k} missing in {npz_path}. Found: {list(data.files)}")
    return {
        "X": data["X_test"],
        "y": data["y_test"],
    }


def load_lipar_npz(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    required = ["X_img_test", "X_seq_test", "y_test"]
    for k in required:
        if k not in data.files:
            raise KeyError(f"{k} missing in {npz_path}. Found: {list(data.files)}")
    return {
        "X_img": data["X_img_test"],
        "X_seq": data["X_seq_test"],
        "y": data["y_test"],
    }


# ============================================================
# Matching logic
# ============================================================
def get_class_indices(y):
    y = np.asarray(y)
    classes = sorted(np.unique(y).tolist())
    idx_map = {}
    for c in classes:
        idx_map[int(c)] = np.where(y == c)[0]
    return idx_map


def compute_common_class_minima(y_a, y_b):
    idx_a = get_class_indices(y_a)
    idx_b = get_class_indices(y_b)

    common = sorted(set(idx_a.keys()).intersection(set(idx_b.keys())))
    if not common:
        raise ValueError("No common classes between the two datasets.")

    per_class_counts = {}
    for c in common:
        m = min(len(idx_a[c]), len(idx_b[c]))
        if m > 0:
            per_class_counts[c] = m

    if not per_class_counts:
        raise ValueError("No overlapping class counts > 0.")

    return per_class_counts


def sample_balanced_indices(y, per_class_counts, seed=42):
    rng = np.random.default_rng(seed)
    idx_map = get_class_indices(y)

    selected = []
    for c, m in per_class_counts.items():
        idx_c = idx_map[c]
        chosen = rng.choice(idx_c, size=m, replace=False)
        selected.append(chosen)

    selected = np.concatenate(selected)
    rng.shuffle(selected)
    return selected


# ============================================================
# Save helpers
# ============================================================
def save_single_vs_single(
    out_path,
    X_a, y_a,
    X_b, y_b,
    name_a, name_b,
    class_counts
):
    np.savez_compressed(
        out_path,
        X_a=X_a,
        y_a=y_a,
        X_b=X_b,
        y_b=y_b,
        model_a=name_a,
        model_b=name_b,
        class_counts_json=json.dumps(class_counts),
    )
    print(f"[SAVED] {out_path}")


def save_single_vs_lipar(
    out_path,
    X_a, y_a,
    X_img_b, X_seq_b, y_b,
    name_a, name_b,
    class_counts
):
    np.savez_compressed(
        out_path,
        X_a=X_a,
        y_a=y_a,
        X_img_b=X_img_b,
        X_seq_b=X_seq_b,
        y_b=y_b,
        model_a=name_a,
        model_b=name_b,
        class_counts_json=json.dumps(class_counts),
    )
    print(f"[SAVED] {out_path}")


# ============================================================
# Pair builders
# ============================================================
def build_lcnn_vs_gru(std_data, out_dir):
    """
    Since both use the same standard NPZ here, we can sample once and reuse
    the same indices on both sides if desired.
    """
    X = std_data["X"]
    y = std_data["y"]

    class_counts = compute_common_class_minima(y, y)
    idx = sample_balanced_indices(y, class_counts, seed=101)

    X_a = X[idx]
    y_a = y[idx]
    X_b = X[idx]
    y_b = y[idx]

    out_path = os.path.join(out_dir, "matched_lcnn_vs_gru.npz")
    save_single_vs_single(
        out_path,
        X_a, y_a,
        X_b, y_b,
        "L-CNN", "GRU",
        class_counts
    )


def build_lcnn_vs_lipar(std_data, lipar_data, out_dir):
    X_std = std_data["X"]
    y_std = std_data["y"]

    X_img = lipar_data["X_img"]
    X_seq = lipar_data["X_seq"]
    y_lip = lipar_data["y"]

    class_counts = compute_common_class_minima(y_std, y_lip)

    idx_std = sample_balanced_indices(y_std, class_counts, seed=202)
    idx_lip = sample_balanced_indices(y_lip, class_counts, seed=203)

    out_path = os.path.join(out_dir, "matched_lcnn_vs_lipar.npz")
    save_single_vs_lipar(
        out_path,
        X_std[idx_std], y_std[idx_std],
        X_img[idx_lip], X_seq[idx_lip], y_lip[idx_lip],
        "L-CNN", "LiPar",
        class_counts
    )


def build_gru_vs_lipar(std_data, lipar_data, out_dir):
    X_std = std_data["X"]
    y_std = std_data["y"]

    X_img = lipar_data["X_img"]
    X_seq = lipar_data["X_seq"]
    y_lip = lipar_data["y"]

    class_counts = compute_common_class_minima(y_std, y_lip)

    idx_std = sample_balanced_indices(y_std, class_counts, seed=303)
    idx_lip = sample_balanced_indices(y_lip, class_counts, seed=304)

    out_path = os.path.join(out_dir, "matched_gru_vs_lipar.npz")
    save_single_vs_lipar(
        out_path,
        X_std[idx_std], y_std[idx_std],
        X_img[idx_lip], X_seq[idx_lip], y_lip[idx_lip],
        "GRU", "LiPar",
        class_counts
    )


# ============================================================
# Optional summary
# ============================================================
def print_npz_summary(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    print(f"\n[SUMMARY] {npz_path}")
    for k in data.files:
        arr = data[k]
        if isinstance(arr, np.ndarray):
            print(f"  - {k}: shape={arr.shape}, dtype={arr.dtype}")
        else:
            print(f"  - {k}: type={type(arr)}")


# ============================================================
# Main
# ============================================================
def main():
    std_data = load_standard_npz(STD_NPZ)
    lipar_data = load_lipar_npz(LIPAR_NPZ)

    print("[INFO] Standard test set:")
    print("  X:", std_data["X"].shape, std_data["X"].dtype)
    print("  y:", std_data["y"].shape, std_data["y"].dtype)

    print("[INFO] LiPar test set:")
    print("  X_img:", lipar_data["X_img"].shape, lipar_data["X_img"].dtype)
    print("  X_seq:", lipar_data["X_seq"].shape, lipar_data["X_seq"].dtype)
    print("  y:", lipar_data["y"].shape, lipar_data["y"].dtype)

    build_lcnn_vs_gru(std_data, OUT_DIR)
    build_lcnn_vs_lipar(std_data, lipar_data, OUT_DIR)
    build_gru_vs_lipar(std_data, lipar_data, OUT_DIR)

    print_npz_summary(os.path.join(OUT_DIR, "matched_lcnn_vs_gru.npz"))
    print_npz_summary(os.path.join(OUT_DIR, "matched_lcnn_vs_lipar.npz"))
    print_npz_summary(os.path.join(OUT_DIR, "matched_gru_vs_lipar.npz"))


if __name__ == "__main__":
    main()

[INFO] Standard test set:
  X: (96762, 1, 20, 10) float32
  y: (96762,) int64
[INFO] LiPar test set:
  X_img: (205640, 3, 9, 9) float32
  X_seq: (205640, 27, 9) float32
  y: (205640,) int64
[SAVED] Prepared/matched_pairs/matched_lcnn_vs_gru.npz
[SAVED] Prepared/matched_pairs/matched_lcnn_vs_lipar.npz
[SAVED] Prepared/matched_pairs/matched_gru_vs_lipar.npz

[SUMMARY] Prepared/matched_pairs/matched_lcnn_vs_gru.npz
  - X_a: shape=(96762, 1, 20, 10), dtype=float32
  - y_a: shape=(96762,), dtype=int64
  - X_b: shape=(96762, 1, 20, 10), dtype=float32
  - y_b: shape=(96762,), dtype=int64
  - model_a: shape=(), dtype=<U5
  - model_b: shape=(), dtype=<U3
  - class_counts_json: shape=(), dtype=<U118

[SUMMARY] Prepared/matched_pairs/matched_lcnn_vs_lipar.npz
  - X_a: shape=(96762, 1, 20, 10), dtype=float32
  - y_a: shape=(96762,), dtype=int64
  - X_img_b: shape=(96762, 3, 9, 9), dtype=float32
  - X_seq_b: shape=(96762, 27, 9), dtype=float32
  - y_b: shape=(96762,), dtype=int64
  - model_a: shape

In [19]:
import os
import json
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import ttest_ind, ttest_rel, mannwhitneyu, wilcoxon

from src.LCNN import LCNN
from src.LiPar import LiPar_STParNet
from src.GRU import GRU


# ============================================================
# Paths
# ============================================================
PAIR_DIR = "Prepared/matched_pairs"

PAIR_FILES = {
    "L-CNN_vs_GRU": os.path.join(PAIR_DIR, "matched_lcnn_vs_gru.npz"),
    "L-CNN_vs_LiPar": os.path.join(PAIR_DIR, "matched_lcnn_vs_lipar.npz"),
    "GRU_vs_LiPar": os.path.join(PAIR_DIR, "matched_gru_vs_lipar.npz"),
}

MODEL_PATHS = {
    "L-CNN": "models/Kia_K_Poin_Mob_Net_20_Over_50.pth",
    "GRU": "models/Kia_GRU_20_Over_50.pth",
    "LiPar": "models/Kia_K_LiPar_STParNet.pt",
}

NUM_CLASSES = 11
BATCH_SIZE = 64


# ============================================================
# Utilities
# ============================================================
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state = checkpoint["model_state_dict"]
        elif "model_state" in checkpoint:
            state = checkpoint["model_state"]
        else:
            state = checkpoint
    else:
        state = checkpoint

    cleaned = {}
    for k, v in state.items():
        if k.startswith("module."):
            cleaned[k[len("module."):]] = v
        else:
            cleaned[k] = v
    return cleaned


def safe_load_checkpoint(ckpt_path, device):
    checkpoint = torch.load(ckpt_path, map_location=device)
    state = extract_state_dict(checkpoint)
    return state


# ============================================================
# Model loaders
# ============================================================
def load_lcnn_model(ckpt_path, device, num_classes=11):
    model = LCNN(num_classes=num_classes).to(device)
    state = safe_load_checkpoint(ckpt_path, device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


def load_gru_model(ckpt_path, device, num_classes=11):
    model = GRU(num_classes=num_classes).to(device)
    state = safe_load_checkpoint(ckpt_path, device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


def load_lipar_model(ckpt_path, device, num_classes=11):
    model = LiPar_STParNet(num_classes=num_classes).to(device)
    state = safe_load_checkpoint(ckpt_path, device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


def load_model_by_name(name, device):
    if name == "L-CNN":
        return load_lcnn_model(MODEL_PATHS[name], device, NUM_CLASSES)
    elif name == "GRU":
        return load_gru_model(MODEL_PATHS[name], device, NUM_CLASSES)
    elif name == "LiPar":
        return load_lipar_model(MODEL_PATHS[name], device, NUM_CLASSES)
    else:
        raise ValueError(f"Unknown model name: {name}")


# ============================================================
# Dataset loaders for matched pair npz
# ============================================================
def load_single_input_dataset(X, y, batch_size=64):
    ds = TensorDataset(
        torch.from_numpy(X).float(),
        torch.from_numpy(y).long()
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=False)


def load_lipar_dataset(X_img, X_seq, y, batch_size=64):
    ds = TensorDataset(
        torch.from_numpy(X_img).float(),
        torch.from_numpy(X_seq).float(),
        torch.from_numpy(y).long()
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=False)


# ============================================================
# Inference helpers
# ============================================================
def get_predictions_and_labels_single_input(model, loader, device, model_name="Model"):
    all_preds = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            x, y = batch[0].to(device), batch[1].to(device)

            if i == 0:
                print(f"[DEBUG] {model_name} first batch x shape: {tuple(x.shape)}")
                print(f"[DEBUG] {model_name} first batch y shape: {tuple(y.shape)}")

            outputs = model(x)
            if isinstance(outputs, (tuple, list)):
                outputs = outputs[0]

            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    corr = (y_pred == y_true).astype(np.int32)
    return y_true, y_pred, corr


def get_predictions_and_labels_lipar(model, loader, device, model_name="LiPar"):
    all_preds = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            x_img = batch[0].to(device)
            x_seq = batch[1].to(device)
            y = batch[2].to(device)

            if i == 0:
                print(f"[DEBUG] {model_name} first batch x_img shape: {tuple(x_img.shape)}")
                print(f"[DEBUG] {model_name} first batch x_seq shape: {tuple(x_seq.shape)}")
                print(f"[DEBUG] {model_name} first batch y shape    : {tuple(y.shape)}")

            outputs = model(x_img, x_seq)
            if isinstance(outputs, (tuple, list)):
                outputs = outputs[0]

            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    corr = (y_pred == y_true).astype(np.int32)
    return y_true, y_pred, corr


# ============================================================
# Statistical tests
# ============================================================
def run_tests(corr_a, corr_b, paired=False):
    results = {}

    # T-test
    if paired:
        _, t_p = ttest_rel(corr_a, corr_b)
        results["t_test_name"] = "Paired t-test"
    else:
        _, t_p = ttest_ind(corr_a, corr_b, equal_var=False)
        results["t_test_name"] = "Independent t-test"

    results["t_test_p"] = float(t_p)

    # Mann-Whitney U
    try:
        _, mw_p = mannwhitneyu(corr_a, corr_b, alternative="two-sided")
    except Exception:
        mw_p = 1.0
    results["mannwhitney_p"] = float(mw_p)

    # Wilcoxon
    # Only truly valid for paired/aligned samples
    try:
        if len(corr_a) == len(corr_b):
            diff = corr_a - corr_b
            if np.all(diff == 0):
                w_p = 1.0
            else:
                _, w_p = wilcoxon(corr_a, corr_b, zero_method="wilcox", alternative="two-sided")
        else:
            w_p = None
    except Exception:
        w_p = None

    results["wilcoxon_p"] = None if w_p is None else float(w_p)
    return results


def p_to_text(p):
    if p is None:
        return "--"
    if p < 0.01:
        return "p < 0.01"
    elif p < 0.05:
        return "p < 0.05"
    else:
        return f"p = {p:.6f}"


def p_to_latex(p):
    if p is None:
        return "--"
    if p < 0.01:
        return r"$p < 0.01$"
    elif p < 0.05:
        return r"$p < 0.05$"
    else:
        return rf"$p = {p:.4f}$"


# ============================================================
# One pair evaluator
# ============================================================
def evaluate_pair(pair_name, pair_path, device):
    print("\n" + "=" * 80)
    print(f"[PAIR] {pair_name}")
    print("=" * 80)

    data = np.load(pair_path, allow_pickle=True)

    model_a = str(data["model_a"])
    model_b = str(data["model_b"])
    class_counts = json.loads(str(data["class_counts_json"]))

    print(f"[INFO] model_a = {model_a}")
    print(f"[INFO] model_b = {model_b}")
    print(f"[INFO] balanced class counts = {class_counts}")

    # Decide if this is true paired or only balanced
    true_paired = (pair_name == "L-CNN_vs_GRU")

    # Load model A
    net_a = load_model_by_name(model_a, device)
    if model_a in ["L-CNN", "GRU"]:
        loader_a = load_single_input_dataset(data["X_a"], data["y_a"], batch_size=BATCH_SIZE)
        y_a, pred_a, corr_a = get_predictions_and_labels_single_input(net_a, loader_a, device, model_name=model_a)
    else:
        raise ValueError("model_a should not be LiPar in these saved pair files.")

    # Load model B
    net_b = load_model_by_name(model_b, device)
    if model_b in ["L-CNN", "GRU"]:
        loader_b = load_single_input_dataset(data["X_b"], data["y_b"], batch_size=BATCH_SIZE)
        y_b, pred_b, corr_b = get_predictions_and_labels_single_input(net_b, loader_b, device, model_name=model_b)
    elif model_b == "LiPar":
        loader_b = load_lipar_dataset(data["X_img_b"], data["X_seq_b"], data["y_b"], batch_size=BATCH_SIZE)
        y_b, pred_b, corr_b = get_predictions_and_labels_lipar(net_b, loader_b, device, model_name=model_b)
    else:
        raise ValueError(f"Unsupported model_b: {model_b}")

    print(f"\n[INFO] {model_a} accuracy = {corr_a.mean():.6f}")
    print(f"[INFO] {model_b} accuracy = {corr_b.mean():.6f}")
    print(f"[INFO] sample count A = {len(corr_a)}")
    print(f"[INFO] sample count B = {len(corr_b)}")

    results = run_tests(corr_a, corr_b, paired=true_paired)

    print(f"\n[RESULT] {model_a} vs {model_b}")
    print(f"  {results['t_test_name']} : {p_to_text(results['t_test_p'])}")
    print(f"  Mann-Whitney U           : {p_to_text(results['mannwhitney_p'])}")
    print(f"  Wilcoxon                : {p_to_text(results['wilcoxon_p'])}")

    if not true_paired:
        print("  [NOTE] Wilcoxon here is numerically computed but not methodologically valid as a true paired test.")

    return {
        "pair_name": pair_name,
        "model_a": model_a,
        "model_b": model_b,
        "paired": true_paired,
        "acc_a": float(corr_a.mean()),
        "acc_b": float(corr_b.mean()),
        "n_a": int(len(corr_a)),
        "n_b": int(len(corr_b)),
        **results
    }


# ============================================================
# Main
# ============================================================
def main():
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Using device: {device}")

    all_results = []
    for pair_name, pair_path in PAIR_FILES.items():
        res = evaluate_pair(pair_name, pair_path, device)
        all_results.append(res)

    print("\n" + "=" * 80)
    print("LaTeX rows")
    print("=" * 80)
    for r in all_results:
        print(
            f"{r['model_a']} & {r['model_b']} & "
            f"{p_to_latex(r['t_test_p'])} & "
            f"{p_to_latex(r['mannwhitney_p'])} & "
            f"{p_to_latex(r['wilcoxon_p'])} \\\\"
        )

    print("\n" + "=" * 80)
    print("Full LaTeX table")
    print("=" * 80)
    print(rf"""
\begin{{table}}[t]
\centering
\caption{{Statistical comparison on matched pairwise subsets.}}
\label{{tab:matched_pairwise_stats}}
\begin{{tabular}}{{l l c c c}}
\toprule
Model A & Model B & T-test & Mann-Whitney U & Wilcoxon \\
\midrule
{all_results[0]['model_a']} & {all_results[0]['model_b']} & {p_to_latex(all_results[0]['t_test_p'])} & {p_to_latex(all_results[0]['mannwhitney_p'])} & {p_to_latex(all_results[0]['wilcoxon_p'])} \\
{all_results[1]['model_a']} & {all_results[1]['model_b']} & {p_to_latex(all_results[1]['t_test_p'])} & {p_to_latex(all_results[1]['mannwhitney_p'])} & {p_to_latex(all_results[1]['wilcoxon_p'])} \\
{all_results[2]['model_a']} & {all_results[2]['model_b']} & {p_to_latex(all_results[2]['t_test_p'])} & {p_to_latex(all_results[2]['mannwhitney_p'])} & {p_to_latex(all_results[2]['wilcoxon_p'])} \\
\bottomrule
\end{{tabular}}
\end{{table}}
""")


if __name__ == "__main__":
    main()

[INFO] Using device: cuda

[PAIR] L-CNN_vs_GRU
[INFO] model_a = L-CNN
[INFO] model_b = GRU
[INFO] balanced class counts = {'0': 71075, '1': 5600, '2': 7140, '3': 5534, '4': 80, '5': 1128, '6': 1449, '7': 3554, '8': 384, '9': 380, '10': 438}
[DEBUG] L-CNN first batch x shape: (64, 1, 20, 10)
[DEBUG] L-CNN first batch y shape: (64,)
[DEBUG] GRU first batch x shape: (64, 1, 20, 10)
[DEBUG] GRU first batch y shape: (64,)

[INFO] L-CNN accuracy = 0.995897
[INFO] GRU accuracy = 0.982038
[INFO] sample count A = 96762
[INFO] sample count B = 96762

[RESULT] L-CNN vs GRU
  Paired t-test : p < 0.01
  Mann-Whitney U           : p < 0.01
  Wilcoxon                : p < 0.01

[PAIR] L-CNN_vs_LiPar
[INFO] model_a = L-CNN
[INFO] model_b = LiPar
[INFO] balanced class counts = {'0': 71075, '1': 5600, '2': 7140, '3': 5534, '4': 80, '5': 1128, '6': 1449, '7': 3554, '8': 384, '9': 380, '10': 438}
[DEBUG] L-CNN first batch x shape: (64, 1, 20, 10)
[DEBUG] L-CNN first batch y shape: (64,)
[DEBUG] LiPar firs